# 01 — PDF Text Extraction

This notebook implements the first stage of the **CLARA** document ingestion pipeline.

Its purpose is to extract text from all PDF files available in the `data/raw/` directory while preserving the original page boundaries. Each document is converted into a structured JSON representation that will be used by the subsequent parsing, chunking, embedding, and indexing stages.

The output generated by this notebook follows the structure:

```text
data/processed/<DOCUMENT_ID>/extracted.json
```

At this stage, no legal structure, article detection, or text chunking is performed. The objective is to obtain a consistent page-level representation of each source document.

## 1. Imports

The notebook uses a small set of libraries for filesystem operations, JSON serialisation, and PDF processing.

- `pathlib` provides platform-independent filesystem paths.
- `json` is used to store the extracted documents in a structured format.
- `PyMuPDF` (`pymupdf`) provides PDF reading and text extraction capabilities.

In [1]:
from pathlib import Path
import json
import pymupdf

## 2. Project Paths

The input and output directories are defined relative to the project root.

The expected directory structure is:

```text
data/
├── raw/          # Original PDF documents
└── processed/    # Intermediate processed representations
```

Using relative project paths avoids hard-coded machine-specific locations and makes the notebook easier to reproduce in different environments.

In [2]:
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DIR, PROCESSED_DIR

(WindowsPath('C:/Users/user/Documents/GitHub/clara-rag-pt/data/raw'),
 WindowsPath('C:/Users/user/Documents/GitHub/clara-rag-pt/data/processed'))

### 2.1 Directory Initialisation

The required input and output directories are created automatically if they do not already exist.

The resolved paths are also displayed so that the working directories can be verified before processing begins.

In [3]:
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw:       {RAW_DIR}")
print(f"Processed: {PROCESSED_DIR}")

Raw:       C:\Users\user\Documents\GitHub\clara-rag-pt\data\raw
Processed: C:\Users\user\Documents\GitHub\clara-rag-pt\data\processed


## 3. PDF Document Discovery

All PDF files located in `data/raw/` are automatically discovered.

This allows the ingestion pipeline to process multiple legal documents without manually configuring individual file paths. The files are sorted to provide a consistent processing order between executions.

The discovered files are displayed before extraction so that the input corpus can be verified.

In [4]:
pdf_files = sorted(RAW_DIR.glob("*.pdf"))

print(f"Foram encontrados {len(pdf_files)} ficheiros PDF:\n")

for pdf_path in pdf_files:
    print(f"- {pdf_path.name}")

Foram encontrados 10 ficheiros PDF:

- CIMI.pdf
- CIRC.pdf
- CIRS.pdf
- CIS.pdf
- CIVA.pdf
- EBF.pdf
- LGT.pdf
- OE2026.pdf
- RGIT.pdf
- RITI.pdf


## 4. Page-Level Text Extraction

The `extract_pdf()` function converts a PDF document into a structured Python dictionary.

Each document is processed page by page. The resulting representation preserves:

- the document identifier;
- the original source filename;
- the total number of pages;
- the page number;
- the text extracted from each page.

Preserving page boundaries is important because page references can later be associated with retrieved legal passages and included in source metadata.

The document identifier is derived from the PDF filename. For example:

```text
CIRS.pdf → CIRS
```

In [5]:
def extract_pdf(pdf_path: Path) -> dict:
    """
    Extrai o texto de um PDF página a página.

    Parameters
    ----------
    pdf_path : Path
        Caminho para o ficheiro PDF.

    Returns
    -------
    dict
        Estrutura contendo metadados do documento
        e o texto extraído de cada página.
    """

    document_id = pdf_path.stem

    pages = []

    with pymupdf.open(pdf_path) as pdf:

        for page_index, page in enumerate(pdf):

            text = page.get_text("text")

            pages.append({
                "page": page_index + 1,
                "text": text.strip()
            })

        result = {
            "document_id": document_id,
            "source_file": pdf_path.name,
            "num_pages": len(pdf),
            "pages": pages
        }

    return result

## 5. Extraction Test

Before processing the complete corpus, the extraction function is tested on the first PDF discovered in the input directory.

This provides an initial validation that the document can be opened and that its identifier and page count are extracted correctly.

In [6]:
test_pdf = pdf_files[0]

print(f"A testar: {test_pdf.name}")

document = extract_pdf(test_pdf)

print(f"Documento: {document['document_id']}")
print(f"Páginas: {document['num_pages']}")

A testar: CIMI.pdf
Documento: CIMI
Páginas: 91


### 5.1 Inspecting Extracted Text

A sample page from the test document is displayed to manually inspect the quality of the extracted text.

This step can reveal issues such as missing text, unexpected line breaks, recurring headers or footers, formatting artefacts, or pages represented mainly as images.

In [7]:
page_number = 1

page = document["pages"][page_number - 1]

print(f"Página {page['page']}")
print("-" * 80)
print(page["text"][:3000])

Página 1
--------------------------------------------------------------------------------



### 5.2 Empty Page Detection

Some PDF pages may not contain extractable text. This can occur when a page is blank, contains only images, or uses a PDF structure from which text cannot be directly extracted.

This validation identifies pages whose extracted text is empty. Detecting them early helps identify potential gaps before parsing, chunking, and vector indexing.

In [8]:
empty_pages = [
    page["page"]
    for page in document["pages"]
    if not page["text"].strip()
]

print(f"Páginas sem texto: {len(empty_pages)}")

if empty_pages:
    print(empty_pages[:50])

Páginas sem texto: 1
[1]


## 6. Saving Extracted Documents

The extracted document is stored as JSON so that subsequent notebooks can operate on a stable intermediate representation without repeatedly reading the original PDF.

Each document is stored in its own directory:

```text
data/processed/<DOCUMENT_ID>/extracted.json
```

For example:

```text
data/processed/CIRS/extracted.json
```

This organisation allows each legal document to be processed independently during later stages.

In [9]:
def save_extracted_document(document: dict, output_dir: Path) -> Path:
    """
    Guarda um documento extraído em JSON.
    """

    document_id = document["document_id"]

    document_dir = output_dir / document_id
    document_dir.mkdir(parents=True, exist_ok=True)

    output_path = document_dir / "extracted.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(
            document,
            f,
            ensure_ascii=False,
            indent=2
        )

    return output_path

### 6.1 Saving the Test Document

The previously extracted test document is written to disk.

This validates both the output directory structure and the JSON serialisation process before the complete corpus is processed.

In [10]:
output_path = save_extracted_document(
    document,
    PROCESSED_DIR
)

print(f"Guardado em:")
print(output_path)

Guardado em:
C:\Users\user\Documents\GitHub\clara-rag-pt\data\processed\CIMI\extracted.json


## 7. Processing the Complete Corpus

After validating the extraction process on a single document, all PDF files discovered in `data/raw/` are processed.

For each document, the pipeline:

1. opens the PDF;
2. extracts text page by page;
3. builds the structured document representation;
4. saves the result as `extracted.json`;
5. records the processing status.

Errors are handled individually so that a problem with one document does not prevent the remaining corpus from being processed.

In [11]:
results = []

for pdf_path in pdf_files:

    print(f"A processar {pdf_path.name}...", end=" ")

    try:

        document = extract_pdf(pdf_path)

        output_path = save_extracted_document(
            document,
            PROCESSED_DIR
        )

        results.append({
            "document_id": document["document_id"],
            "file": pdf_path.name,
            "pages": document["num_pages"],
            "output": str(output_path),
            "status": "OK"
        })

        print(
            f"OK ({document['num_pages']} páginas)"
        )

    except Exception as e:

        results.append({
            "document_id": pdf_path.stem,
            "file": pdf_path.name,
            "pages": None,
            "output": None,
            "status": f"ERROR: {e}"
        })

        print(f"ERRO: {e}")

A processar CIMI.pdf... OK (91 páginas)
A processar CIRC.pdf... OK (164 páginas)
A processar CIRS.pdf... OK (139 páginas)
A processar CIS.pdf... OK (64 páginas)
A processar CIVA.pdf... OK (188 páginas)
A processar EBF.pdf... OK (107 páginas)
A processar LGT.pdf... OK (65 páginas)
A processar OE2026.pdf... OK (270 páginas)
A processar RGIT.pdf... OK (73 páginas)
A processar RITI.pdf... OK (74 páginas)


## 8. Processing Results

The processing status of each document is displayed together with its document identifier and number of pages.

This provides a quick overview of whether every PDF in the corpus was successfully extracted.

In [12]:
for result in results:

    print(
        f"{result['document_id']:10} "
        f"| páginas: {str(result['pages']):5} "
        f"| {result['status']}"
    )

CIMI       | páginas: 91    | OK
CIRC       | páginas: 164   | OK
CIRS       | páginas: 139   | OK
CIS        | páginas: 64    | OK
CIVA       | páginas: 188   | OK
EBF        | páginas: 107   | OK
LGT        | páginas: 65    | OK
OE2026     | páginas: 270   | OK
RGIT       | páginas: 73    | OK
RITI       | páginas: 74    | OK


### 8.1 Corpus Summary

A global summary is calculated after extraction.

It reports the number of successfully processed documents, the total number of input documents, and the total number of pages extracted from successful documents.

This provides a simple integrity check for the first stage of the ingestion pipeline.

In [13]:
total_documents = len(results)

successful = [
    r for r in results
    if r["status"] == "OK"
]

total_pages = sum(
    r["pages"]
    for r in successful
    if r["pages"] is not None
)

print(f"Documentos processados: {len(successful)}/{total_documents}")
print(f"Total de páginas: {total_pages}")

Documentos processados: 10/10
Total de páginas: 1235


## 9. Corpus-Level Empty Page Validation

After all documents have been saved, the generated `extracted.json` files are reopened and checked for pages without extracted text.

This corpus-level validation complements the earlier single-document test and helps identify documents that may require additional inspection or alternative extraction methods.

In [14]:
empty_pages_summary = []

for result in successful:

    document_id = result["document_id"]

    json_path = (
        PROCESSED_DIR
        / document_id
        / "extracted.json"
    )

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    empty_pages = [
        page["page"]
        for page in data["pages"]
        if not page["text"].strip()
    ]

    empty_pages_summary.append({
        "document_id": document_id,
        "num_pages": data["num_pages"],
        "empty_pages": len(empty_pages)
    })

### 9.1 Empty Page Summary

The final validation reports the total number of pages and the number of empty pages detected for each document.

Documents with unexpected empty pages should be reviewed before continuing to the parsing stage.

## Next Step

The extracted JSON files generated by this notebook are the input to:

```text
02_document_parsing.ipynb
```

The next notebook analyses the extracted text and identifies legal structures such as articles, article titles, and document hierarchy.

In [15]:
for item in empty_pages_summary:

    print(
        f"{item['document_id']:10} "
        f"| páginas: {item['num_pages']:4} "
        f"| vazias: {item['empty_pages']:3}"
    )

CIMI       | páginas:   91 | vazias:   1
CIRC       | páginas:  164 | vazias:   1
CIRS       | páginas:  139 | vazias:   1
CIS        | páginas:   64 | vazias:   1
CIVA       | páginas:  188 | vazias:   1
EBF        | páginas:  107 | vazias:   1
LGT        | páginas:   65 | vazias:   1
OE2026     | páginas:  270 | vazias:   0
RGIT       | páginas:   73 | vazias:   1
RITI       | páginas:   74 | vazias:   1
